# Flipkart Product Review Sentiment

There isn't a clean public Flipkart review dataset that I could find — the closest mirrors all looked scraped and a few were behind a Kaggle login. To make this exercise reproducible I generated a synthetic 500-row corpus of Flipkart-style reviews (short, India-specific phrasing, mentions like "rupees" and Indian product categories). The labels are positive / negative.

If/when I get access to the real Flipkart scrape, only the loader cell needs to change — the model code stays the same. I want to be upfront about this: the numbers below are not on real Flipkart data.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

np.random.seed(7)
DATA_DIR = Path("../../data/flipkart_sentiment")


In [2]:
df = pd.read_csv(DATA_DIR / "flipkart_reviews.csv")
print(df.shape)
print(df.sentiment.value_counts())
df.head()


(500, 2)
sentiment
negative    250
positive    250
Name: count, dtype: int64


,review,sentiment
0,"Camera quality is frustrating, photos look was...",negative
1,"Camera quality is awful, photos look washed out.",negative
2,Heating issue right out of the box.,negative
3,"Worth every rupee, superb product.",positive
4,"Waste of money, awful charger.",negative


## TF-IDF baseline + LinearSVC

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"].values, df["sentiment"].values,
    test_size=0.2, stratify=df["sentiment"], random_state=7
)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

# tried logreg first then switched to LinearSVC just because the text is short and
# the margin-based loss does slightly better on these tiny snippets in my experience.
clf = LinearSVC()
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)

print(classification_report(y_test, pred))


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        50
    positive       1.00      1.00      1.00        50

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



## Notes

Because this is synthetic data the model is going to look much better than it would on the real thing. Synthetic reviews use a small vocabulary and the polarity words are dead obvious. Don't read the precision/recall numbers as anything other than "the pipeline runs".

The realistic failure modes on actual Flipkart data would be:
- Mixed-language reviews (Hinglish), where TF-IDF tokenization gets weird.
- Sarcasm and qualifiers ("works great after I returned it twice").
- Star-rating inconsistency — people sometimes give 5 stars with a negative-sounding review.

For a real production version of this I'd want a fine-tuned multilingual model (XLM-R or muril for Indic content). Out of scope here.
